In [ ]:
# =========================
# FULL PIPELINE (SUBMISSION VERSION) — UPDATED (GLOBAL + SPLITS DISTRIBUTION)
# Arabic Hate Speech Detection
# - Cleaning + Duplicate removal (on clean_text)
# - Interpret hate/non-hate distribution for:
#     ✅ Full dataset (after cleaning+dedup)
#     ✅ Train / Val / Test splits
# - Train/Val/Test split
# - Tokenizer + Padding
# - Baseline TF-IDF + Logistic Regression
# - FastText Arabic embeddings (cc.ar.300.vec)
# - LSTM + BiLSTM + CNN+BiLSTM
# - tf.data.Dataset
# - Threshold tuning on validation set
# - Results table + confusion matrices
# =========================

# ---- 0) Imports ----
import os, re, random, numpy as np, pandas as pd
import tensorflow as tf

# ---- 1) Reproducibility ----
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---- 2) Config ----
TEXT_COLUMN  = "text"
LABEL_COLUMN = "label"
DATA_PATH = "/content/Arabic_Tweets_dataset.csv"

MAX_VOCAB = 30000
MAX_LEN   = 100
EMBEDDING_DIM = 300

EPOCHS = 8
BATCH_SIZE = 64
LSTM_UNITS = 64

FASTTEXT_GZ  = "cc.ar.300.vec.gz"
FASTTEXT_VEC = "cc.ar.300.vec"
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.vec.gz"

# ---- 3) Cleaning ----
def clean_arabic(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[\u064B-\u0652\u0670\u0653-\u065F]", "", text)   # diacritics
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)                  # keep Arabic letters + space
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                       # long repeats -> 2 chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---- 3.1) Remove duplicates ----
def remove_duplicate_tweets(df, text_col="clean_text", keep="first", verbose=True):
    before = len(df)
    df_dedup = df.drop_duplicates(subset=[text_col], keep=keep).reset_index(drop=True)
    after = len(df_dedup)
    removed = before - after

    if verbose:
        print("🧹 Duplicate tweet cleanup")
        print(f"   Before: {before:,}")
        print(f"   After : {after:,}")
        print(f"   Removed duplicates: {removed:,} ({removed/before:.2%})")
        remaining_dups = df_dedup.duplicated(subset=[text_col]).sum()
        print(f"   Remaining duplicates (should be 0): {remaining_dups}")
    return df_dedup

# ---- 3.2) Fun: interpret hate/non-hate counts ----
def interpret_class_distribution(hate_count, non_hate_count, title="Class Distribution"):
    total = hate_count + non_hate_count
    hate_pct = (hate_count / total) * 100 if total else 0
    non_hate_pct = (non_hate_count / total) * 100 if total else 0

    print(f"\n📊 {title}")
    print("-" * 48)
    print(f"🧮 Total samples     : {total:,}")
    print(f"🔥 Hate tweets       : {hate_count:,} ({hate_pct:.2f}%)")
    print(f"🌱 Non-hate tweets   : {non_hate_count:,} ({non_hate_pct:.2f}%)")

    if total == 0:
        print("⚠️  No samples to interpret.\n")
        return

    if abs(hate_pct - non_hate_pct) < 5:
        print("⚖️  Balanced dataset (roughly).")
    elif hate_pct > non_hate_pct:
        print("⚠️  Moderately skewed toward HATE.")
    else:
        print("⚠️  Moderately skewed toward NON-HATE.")

    print("💡 This supports using F1-score (and precision/recall) over accuracy.\n")

def counts_from_labels(y_series):
    """Returns (hate_count, non_hate_count) from label series where 1=hate, 0=non-hate."""
    hate = int((y_series == 1).sum())
    non_hate = int((y_series == 0).sum())
    return hate, non_hate

# ---- 4) Load + clean dataset ----
df = pd.read_csv(DATA_PATH)[[TEXT_COLUMN, LABEL_COLUMN]].copy()
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)

df[LABEL_COLUMN] = pd.to_numeric(df[LABEL_COLUMN], errors="coerce")
df = df.dropna(subset=[LABEL_COLUMN]).reset_index(drop=True)
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

df["clean_text"] = df[TEXT_COLUMN].apply(clean_arabic)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

# ✅ Remove duplicates (ACTUALLY apply)
df = remove_duplicate_tweets(df, text_col="clean_text", keep="first", verbose=True)

# ---- 4.1) Interpret distribution for FULL dataset (after cleaning+dedup) ----
hate_all, non_hate_all = counts_from_labels(df[LABEL_COLUMN])
interpret_class_distribution(hate_all, non_hate_all, title="Full Dataset (after cleaning + dedup)")

X = df["clean_text"]
y = df[LABEL_COLUMN]

# ---- 5) Split train/val/test ----
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=SEED, stratify=y_trainval
)

# ---- 5.1) Interpret distribution for EACH split ----
hate_train, non_hate_train = counts_from_labels(y_train)
hate_val, non_hate_val     = counts_from_labels(y_val)
hate_test, non_hate_test   = counts_from_labels(y_test)

interpret_class_distribution(hate_train, non_hate_train, title="Train Split (ground truth)")
interpret_class_distribution(hate_val, non_hate_val, title="Validation Split (ground truth)")
interpret_class_distribution(hate_test, non_hate_test, title="Test Split (ground truth)")

# ---- 6) Tokenizer + Padding ----
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(tokenizer.texts_to_sequences(X_val),   maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding="post", truncating="post")

# ---- 6.5) tf.data.Dataset ----
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train.values))
    .shuffle(len(X_train_pad), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val.values))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTOTUNE)
)

# ---- 7) Baseline: TF-IDF + Logistic Regression ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

baseline_model = LogisticRegression(max_iter=2000, class_weight="balanced")
baseline_model.fit(X_train_tfidf, y_train)
y_pred_base = baseline_model.predict(X_test_tfidf)

# ---- 8) FastText embeddings ----
def download_fasttext_if_needed():
    if os.path.exists(FASTTEXT_VEC):
        return
    import subprocess
    subprocess.run(["wget", "-q", FASTTEXT_URL, "-O", FASTTEXT_GZ], check=True)
    subprocess.run(["gunzip", "-f", FASTTEXT_GZ], check=True)

download_fasttext_if_needed()

word_index = tokenizer.word_index
vocab_size = min(MAX_VOCAB, len(word_index) + 1)
needed_words = {w for w, i in word_index.items() if i < vocab_size}

embedding_index = {}
with open(FASTTEXT_VEC, "r", encoding="utf-8", errors="ignore") as f:
    next(f)  # header
    for line in f:
        parts = line.rstrip().split(" ")
        w = parts[0]
        if w in needed_words:
            embedding_index[w] = np.asarray(parts[1:], dtype="float32")

embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM), dtype="float32")
for w, i in word_index.items():
    if i < vocab_size and w in embedding_index:
        embedding_matrix[i] = embedding_index[w]

# ---- 9) Models ----
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout, Conv1D, MaxPooling1D, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6)

def build_lstm():
    m = tf.keras.Sequential([
        Embedding(vocab_size, EMBEDDING_DIM, weights=[embedding_matrix], trainable=True, mask_zero=True),
        SpatialDropout1D(0.2),
        LSTM(128, dropout=0.2, recurrent_dropout=0.2),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

def build_bilstm():
    m = Sequential([
        Input(shape=(MAX_LEN,)),
        Embedding(vocab_size, EMBEDDING_DIM, weights=[embedding_matrix], trainable=False),
        Bidirectional(LSTM(LSTM_UNITS)),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

def build_cnn_bilstm():
    m = Sequential([
        Input(shape=(MAX_LEN,)),
        Embedding(vocab_size, EMBEDDING_DIM, weights=[embedding_matrix], trainable=False),
        Conv1D(128, 5, activation="relu", padding="same"),
        MaxPooling1D(2),
        Bidirectional(LSTM(LSTM_UNITS)),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

model_lstm  = build_lstm()
model_bilstm = build_bilstm()
model_cnn    = build_cnn_bilstm()

# ---- 10) Train ----
model_lstm.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop, reduce_lr])
model_bilstm.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop, reduce_lr])
model_cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop, reduce_lr])

# ---- 11) Threshold tuning ----
def find_best_threshold(model, X_val_pad, y_val):
    y_prob = model.predict(X_val_pad, batch_size=64).ravel()
    best_thr, best_f1 = 0.5, -1
    for thr in np.arange(0.1, 0.91, 0.01):
        y_pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr

thr_lstm   = find_best_threshold(model_lstm,  X_val_pad, y_val)
thr_bilstm = find_best_threshold(model_bilstm, X_val_pad, y_val)
thr_cnn    = find_best_threshold(model_cnn,   X_val_pad, y_val)

print("\nSizes after deduplication:")
print("Total:", len(df))
print("Train:", len(X_train), f"({len(X_train)/len(df):.2%})")
print("Val:  ", len(X_val),   f"({len(X_val)/len(df):.2%})")
print("Test: ", len(X_test),  f"({len(X_test)/len(df):.2%})")

# ---- 12) Evaluation ----
def eval_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"Model": name, "Accuracy": acc, "Precision": p, "Recall": r, "F1": f1}

results = []

# Baseline
results.append(eval_model("TF-IDF + Logistic Regression", y_test, y_pred_base))
print("\nTF-IDF + Logistic Regression Confusion Matrix:\n", confusion_matrix(y_test, y_pred_base))

# Deep models
best_model_name = None
best_f1 = -1
best_cm = None

for name, model, thr in [
    ("FastText + LSTM", model_lstm, thr_lstm),
    ("FastText + BiLSTM", model_bilstm, thr_bilstm),
    ("FastText + CNN + BiLSTM", model_cnn, thr_cnn)
]:
    probs = model.predict(X_test_pad, batch_size=64).ravel()
    preds = (probs >= thr).astype(int)

    cm = confusion_matrix(y_test, preds)
    print(f"\n{name} Confusion Matrix:\n", cm)

    metrics = eval_model(name, y_test, preds)
    results.append(metrics)

    if metrics["F1"] > best_f1:
        best_f1 = metrics["F1"]
        best_model_name = name
        best_cm = cm

# Results table
df_results = pd.DataFrame(results).sort_values("F1", ascending=False)
print("\nResults (sorted by F1):")
print(df_results)

print(f"\n🏆 Best model by F1: {best_model_name} (F1={best_f1:.4f})")
print("\nDone ✅")


🧹 Duplicate tweet cleanup
   Before: 13,863
   After : 12,841
   Removed duplicates: 1,022 (7.37%)
   Remaining duplicates (should be 0): 0

📊 Full Dataset (after cleaning + dedup)
------------------------------------------------
🧮 Total samples     : 12,841
🔥 Hate tweets       : 7,540 (58.72%)
🌱 Non-hate tweets   : 5,301 (41.28%)
⚠️  Moderately skewed toward HATE.
💡 This supports using F1-score (and precision/recall) over accuracy.


📊 Train Split (ground truth)
------------------------------------------------
🧮 Total samples     : 8,217
🔥 Hate tweets       : 4,825 (58.72%)
🌱 Non-hate tweets   : 3,392 (41.28%)
⚠️  Moderately skewed toward HATE.
💡 This supports using F1-score (and precision/recall) over accuracy.


📊 Validation Split (ground truth)
------------------------------------------------
🧮 Total samples     : 2,055
🔥 Hate tweets       : 1,207 (58.73%)
🌱 Non-hate tweets   : 848 (41.27%)
⚠️  Moderately skewed toward HATE.
💡 This supports using F1-score (and precision/recall) ove